In [30]:
from transformers import BertModel, BertTokenizer
import torch
from sklearn.metrics.pairwise import cosine_similarity

In [31]:
model = BertModel.from_pretrained('bert-base-uncased')

In [32]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

C:\ProgramData\anaconda3\envs\llms\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [4]:
tokens = tokenizer.encode('select from users')
tokens

[101, 7276, 2013, 5198, 102]

In [5]:
tokens2 = tokenizer.encode('select * from users')
tokens2

[101, 7276, 1008, 2013, 5198, 102]

In [6]:
response = model(torch.tensor(tokens).unsqueeze(0))
print(response.last_hidden_state)

tensor([[[ 0.0708,  0.2150, -0.2355,  ..., -0.3778, -0.0739,  0.6075],
         [ 0.3861, -0.3637,  0.2633,  ..., -0.0226,  0.2078,  0.4141],
         [ 0.2413, -0.8393,  0.3881,  ..., -0.4712, -0.2357,  0.5790],
         [-0.4915, -0.6035,  0.0991,  ..., -0.0149, -0.1811,  0.3670],
         [ 0.9264,  0.0620, -0.1968,  ...,  0.3409, -0.6607, -0.3727]]],
       grad_fn=<NativeLayerNormBackward0>)


In [7]:
response2 = model(torch.tensor(tokens2).unsqueeze(0))
print(response2.last_hidden_state)

tensor([[[ 0.1190,  0.1229, -0.2660,  ..., -0.1531, -0.1417,  0.5372],
         [ 0.4630, -0.3795,  0.4076,  ..., -0.1689, -0.1909,  0.2679],
         [ 0.8468,  0.1353,  0.7231,  ..., -0.0817, -0.6273,  0.0841],
         [ 0.4740, -0.3561,  0.6088,  ..., -0.0444, -0.4318,  0.4628],
         [-0.1671, -0.3498, -0.0828,  ..., -0.0309, -0.2110,  0.4171],
         [ 0.8761,  0.1303, -0.2154,  ...,  0.5015, -0.7575, -0.4591]]],
       grad_fn=<NativeLayerNormBackward0>)


In [8]:
"$" in tokenizer.vocab

True

# Check vector representation differences between similar words when the context change (Python the snake, python the programming language)

In [9]:
from transformers import BertModel, BertTokenizer
import torch
from sklearn.metrics.pairwise import cosine_similarity

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
print(f'length of BERT base vocabulary: {len(tokenizer.vocab)}')
text = "A simple sentence"
tokens = tokenizer.encode(text)
print(tokens)
tokenizer.decode(tokens)

length of BERT base vocabulary: 30522
[101, 1037, 3722, 6251, 102]


'[CLS] a simple sentence [SEP]'

In [10]:
python_pet = tokenizer.encode("I love my pet python")
python_lang = tokenizer.encode("I love coding in python")
python_pet_embedding = model(torch.tensor(python_pet).unsqueeze(0))[0][:,5,:].detach().numpy()
python_lang_embedding = model(torch.tensor(python_lang).unsqueeze(0))[0][:,5,:].detach().numpy()

snake_alone_embed = model(torch.tensor(tokenizer.encode('snake')).unsqueeze(0))[0][:,1,:].detach()
programming_alone_embed = model(torch.tensor(tokenizer.encode('programming')).unsqueeze(0))[0][:,1,:].detach()

In [11]:
cosine_similarity(python_lang_embedding, snake_alone_embed)

array([[0.5843482]], dtype=float32)

In [12]:
cosine_similarity(python_pet_embedding, snake_alone_embed)

array([[0.6928661]], dtype=float32)

In [17]:
ntok = tokenizer.encode("My dog is cute.", "He likes playing.")
tokenizer.decode(ntok)

'[CLS] my do is cute. [SEP] he likes playing. [SEP]'

In [18]:
model.embeddings

BertEmbeddings(
  (word_embeddings): Embedding(30522, 768, padding_idx=0)
  (position_embeddings): Embedding(512, 768)
  (token_type_embeddings): Embedding(2, 768)
  (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
  (dropout): Dropout(p=0.1, inplace=False)
)

In [33]:
ntok = tokenizer.encode("I am sinan", return_tensors='pt')
model.embeddings.word_embeddings(ntok)

tensor([[[ 0.0136, -0.0265, -0.0235,  ...,  0.0087,  0.0071,  0.0151],
         [-0.0211,  0.0059, -0.0179,  ...,  0.0163,  0.0122,  0.0073],
         [-0.0437, -0.0150,  0.0029,  ..., -0.0282,  0.0474, -0.0448],
         [-0.0022, -0.0876,  0.0143,  ...,  0.0232, -0.0024, -0.0213],
         [-0.0614, -0.0044, -0.0755,  ..., -0.0522, -0.0310, -0.0248],
         [-0.0145, -0.0100,  0.0060,  ..., -0.0250,  0.0046, -0.0015]]],
       grad_fn=<EmbeddingBackward0>)

In [34]:
ntok2 = tokenizer.encode("I am mat", return_tensors='pt')
model.embeddings.word_embeddings(tokenizer.encode("I am mat", return_tensors='pt'))

tensor([[[ 1.3630e-02, -2.6490e-02, -2.3503e-02,  ...,  8.6805e-03,
           7.1340e-03,  1.5147e-02],
         [-2.1087e-02,  5.9040e-03, -1.7926e-02,  ...,  1.6269e-02,
           1.2201e-02,  7.2568e-03],
         [-4.3735e-02, -1.5029e-02,  2.8998e-03,  ..., -2.8239e-02,
           4.7383e-02, -4.4826e-02],
         [-4.9972e-02, -2.7253e-02,  1.3591e-02,  ..., -4.6719e-03,
          -1.8902e-02,  2.1780e-05],
         [-1.4521e-02, -9.9615e-03,  6.0263e-03,  ..., -2.5035e-02,
           4.6379e-03, -1.5378e-03]]], grad_fn=<EmbeddingBackward0>)

# Pre-training and fine-tning BERT models